# 22 - Pass 11: where does the seed recipe's gain come from? (one commit, GPU)

**The question.** The seed paper's recipe is GAN-BERT plus sentiment/emotion helper heads (our re-implementation: **A5**).
This notebook compares it with what we found works (**A36**: MLM pretraining + helper heads + FGM + self-training)
across label budgets, and checks the objections a reviewer will raise first.

| Block | Runs | What it answers |
|---|---|---|
| **70% labels** (seed paper's regime, new budget): A5, A7 (FGM), A1, A20 (MLM) | 20 | Does the seed paper's accuracy range need the GAN or the helper heads? |
| **A36 at 5% and 30%** (A36 at 1% and 10% exists; A5, A1, A20 exist at 5/30%) | 10 | Our recipe vs the seed recipe across the curve |
| **GAN-BERT, 35x steps** (about 10 passes over U), paired with BERT at the same steps | 10 | Is the GAN null caused by too few passes over U? |
| **MLM dose curve**: pretrain for 100 / 350 / 1,000 / 3,000 steps, then A20 at 1% | 20 | Is MLM's advantage a matter of exposure to U (the GAN sees 6,400 rows at 1%; 100 steps x 64 = 6,400)? |
| **A37**: seed recipe (GAN + helpers) on the MLM encoder, 10% and 1% | 10 | Does the seed recipe add anything once the encoder is pretrained? |

**70 runs in total, about 8 GPU-hours (about 4.5 h on T4 x2; estimates from earlier run times).** Everything is resumable: if the session stops, run the run cell again; finished runs are skipped.

**Pre-registration.** Predictions are in `vault/Pass 11 Predictions.md`. The notebook refuses to start unless that file says `status: FROZEN`
in the cloned commit (set `REQUIRE_FROZEN = False` below only for a throw-away look; the results then count as exploratory and the paper must say so).
Nothing is tuned: every setting (lr 5e-5, helper weight 0.3, FGM epsilon 0.5, 200-step floor) is frozen from earlier passes.

**Before you run, in the right-hand panel:**
- **Input 1:** the same `gainsplit-processed` version as notebooks 10-21.
- **Input 2:** the output of **notebook 15** (Add Input -> Your Work -> Notebooks -> `15-pass8-mlm`); it holds the seed-0 MLM encoder (`ckpt/mlm`).
- **Internet: on**, **Accelerator: GPU T4 x2**.
- Then **Save Version -> Save & Run All (Commit)**; you can close the browser.

## 1. Get the code
Clones the public repo and prints the **commit**. Every run records it in its `result.json`.

In [ ]:
import os, subprocess, sys, time, json, shutil, threading
from pathlib import Path
T_START = time.time()
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

## 2. Tests (about three minutes)
Tiny random BERT, made-up data. Pass 11 adds tests for the 70% budget file, the plans, A37 and the analysis. If anything fails, stop and paste the output.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed: see output above"

## 3. Pre-registration check
The predictions must be committed and marked FROZEN before any run. The check reads the cloned commit, so the order is auditable.

In [ ]:
REQUIRE_FROZEN = True
pred = Path(f"{REPO_DIR}/vault/Pass 11 Predictions.md").read_text(encoding="utf-8")
status = [l.split(":", 1)[1].strip() for l in pred.splitlines()[:8] if l.startswith("status:")]
print("predictions status:", status)
if REQUIRE_FROZEN:
    assert status == ["FROZEN"], "vault/Pass 11 Predictions.md is not FROZEN in this commit: freeze, commit and push it first"
else:
    print("WARNING: running without frozen predictions; results are exploratory")

## 4. Find the MLM encoder from notebook 15 and check it is the same one
A36, A20 and A37 must start from **exactly the encoder Pass 8a used**. The cell compares its weight hashes with the committed receipt and stops if it is missing or different.

In [ ]:
from gainsplit.ssl_pretrain import find_encoder
receipt = json.loads(Path(f"{REPO_DIR}/results/pass8_mlm/pretrain_info.json").read_text())
ENC = str(find_encoder("/kaggle/input", "mlm", seed=0, steps=10000, expected_sha256=receipt["weights_sha256"]))
print("MLM encoder (seed 0, 10,000 steps):", ENC, "- weights match the Pass 8a receipt")

## 5. Data and the 70% label budget
`budgets.json` has 1/5/10/30/100%. The seed paper labeled 70% of its training split, so this cell draws **70% sets with the same
procedure and seeds as the frozen file** and checks that (a) the procedure reproduces the frozen 30% sets exactly and (b) each 70% set contains
the 30% set and lies inside the 100% set (nested, as all other budgets are). The new file goes to `/kaggle/working/extra/` and is read through the
`GAINSPLIT_EXTRA_BUDGETS` variable; the frozen data folder is not touched. It is saved in the results zip as a receipt.

In [ ]:
import torch
from gainsplit.data import find_data_dir, verify, make_extra_budgets, load_splits, EXTRA_BUDGETS_ENV
DATA_DIR = find_data_dir("/kaggle/input")
verify(DATA_DIR)
print("data:", DATA_DIR, "- all files match manifest.json")
extra = make_extra_budgets(DATA_DIR, (70,))
Path("/kaggle/working/extra").mkdir(parents=True, exist_ok=True)
EXTRA_PATH = "/kaggle/working/extra/budgets_extra.json"
Path(EXTRA_PATH).write_text(json.dumps(extra))
os.environ[EXTRA_BUDGETS_ENV] = EXTRA_PATH
for s in range(5):
    sp = load_splits(DATA_DIR, 70, s)
    print(f"seed {s}: 70% -> |L|={len(sp.labeled)} |U|={len(sp.unlabeled)}, sarcastic share of L {sp.labeled['label'].mean():.3f}")
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 6. The list of runs
Switch a block off to skip it. Order: longest runs first so the two GPUs stay busy to the end.

In [ ]:
from gainsplit.plans import PLANS, PASS11_CKPT, PASS11_DOSE_STEPS
RUN_HIGH70, RUN_OURS, RUN_GANLONG, RUN_A37, RUN_DOSE = True, True, True, True, True
PLAN = "pass11_core"
OUT_DIR = f"/kaggle/working/results/{PLAN}"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

def keep(c):
    if c.budget == 70: return RUN_HIGH70
    if c.arm == "A36": return RUN_OURS
    if c.tag == "long35": return RUN_GANLONG
    if c.arm == "A37": return RUN_A37
    if c.tag in PASS11_DOSE_STEPS: return RUN_DOSE
    raise ValueError(c.run_id)

configs = [c for c in PLANS["pass11_all"](ENC, PASS11_CKPT) if keep(c)]
for c in configs[:3] + configs[-3:]:
    print(f"{c.run_id:24s} arm={c.arm:4s} labels={c.budget:3d}% seed={c.seed} step_mult={c.step_mult} init={c.init_dir or '-'}")
print(len(configs), "runs")
# CPU pre-flight: every (budget, seed) resolves to a label set; every run is a test-scored run (no tuning here)
for b, s in sorted({(c.budget, c.seed) for c in configs}):
    load_splits(DATA_DIR, b, s)
assert all(c.eval_test for c in configs)

## 7. MLM dose curve: pretrain four short encoders (about 12 minutes)
MLM on the 22,800 training headlines, **no labels**, exactly Pass 8a's settings except the number of steps (100 / 350 / 1,000 / 3,000; seed 0).
100 steps x 64 = 6,400 rows = the GAN's exposure to U at 1% labels. Two GPUs: one runs the three short ones in turn, the other the 3,000-step one.
Finished pretrainings are skipped on a restart. Receipts (`pretrain_info_<tag>.json`) go into the zip.

In [ ]:
def pretrain_chain(gpu, tags):
    for tag in tags:
        env = dict(os.environ, PYTHONPATH=f"{REPO_DIR}/src", CUDA_VISIBLE_DEVICES=str(gpu))
        with open(f"{OUT_DIR}/pretrain_{tag}.log", "w") as log:
            p = subprocess.run([sys.executable, "-m", "gainsplit.ssl_pretrain", "--objective", "mlm", "--data", str(DATA_DIR),
                                "--out", PASS11_CKPT[tag], "--config", json.dumps({"seed": 0, "steps": PASS11_DOSE_STEPS[tag]})],
                               stdout=log, stderr=subprocess.STDOUT, env=env)
        assert p.returncode == 0, f"pretraining {tag} failed: see {OUT_DIR}/pretrain_{tag}.log"
        shutil.copy(f"{PASS11_CKPT[tag]}/pretrain_info.json", f"{OUT_DIR}/pretrain_info_{tag}.json")

if RUN_DOSE:
    t0 = time.time()
    if torch.cuda.device_count() > 1:
        chains = {0: ["dose100", "dose350", "dose1000"], 1: ["dose3000"]}
    else:
        chains = {0: ["dose100", "dose350", "dose1000", "dose3000"]}
    errors = []
    def _run(g, tags):
        try:
            pretrain_chain(g, tags)
        except Exception as e:
            errors.append(repr(e))
    threads = [threading.Thread(target=_run, args=(g, t)) for g, t in chains.items()]
    for t in threads: t.start()
    for t in threads: t.join()
    assert not errors, errors
    for tag in PASS11_DOSE_STEPS:
        info = json.loads(Path(f"{OUT_DIR}/pretrain_info_{tag}.json").read_text())
        print(f"{tag}: {info['minutes']} min; dev MLM loss {info['dev_before']['loss']} -> {info['dev_after']['loss']}; "
              f"probe 1% {info['probe_before']['p1']['dev_macro_f1_mean']:.2f} -> {info['probe_after']['p1']['dev_macro_f1_mean']:.2f}")
    print(f"pretraining: {(time.time() - t0) / 60:.1f} min")

## Run
One run per GPU; when a GPU frees up, the next starts. Each run's log is in `results/pass11_core/logs/<run_id>.log`.
If the session stops halfway, run this cell again: finished runs are skipped.

In [ ]:
from gainsplit.runner import run_all
table = run_all(configs, DATA_DIR, OUT_DIR)
table.to_csv(f"{OUT_DIR}/summary.csv", index=False)
print(f"total so far: {(time.time() - T_START) / 60:.1f} min")

## Checks and analysis (CPU, about 5 minutes)
Paired bootstrap over test headlines (10,000 resamples, same resample for every seed, mean over seeds 0-4), Holm inside each table.
Existing runs of A1, A4, A5, A7, A20, A36 come from the committed results folders in the clone (Passes 1, 2a, 2b, 8a, 10).
Tables: `ours_vs_seed`, `seed_parts`, `high_f1`, `high_accuracy`, `gan_long`, `dose`, `a37`, plus reading aids `summary`, `dose_summary`, `high_summary`.

In [ ]:
from gainsplit import analysis_pass11 as a11
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
RESULTS_DIRS = [d for d in [f"{REPO_DIR}/results/{p}" for p in ("pass1", "pass2a", "pass2b", "pass8_mlm", "pass10_stack", "pass10_seeds")]
                if Path(d).exists()] + [OUT_DIR]
print("results folders:", RESULTS_DIRS)
tables = a11.run_all(RESULTS_DIRS, f"{OUT_DIR}/analysis")
for name, t in tables.items():
    if name != "verdicts":
        print(f"\n=== {name}")
        display(t)

## Verdict against the predictions written before running
"Supported" needs the predicted direction **and** p_holm < 0.05; "no difference" predictions are supported when p_holm >= 0.05 (not proof of equality: read the interval).
`not+` means "not better than": supported unless the difference is significantly positive. Paste this cell's output when you report.

In [ ]:
v = tables["verdicts"]
for _, r in v.iterrows():
    if pd.isna(r.get("mean_diff")):
        print(f"{r.hypothesis} {r.contrast} at {r.budget}%: {r.verdict}")
        continue
    print(f"{r.hypothesis} {r.contrast:62s} {r.budget:3d}%  {r.mean_diff:+.2f} [{r.ci_low:+.2f}, {r.ci_high:+.2f}]  p_holm {r.p_holm:.3f}  -> {r.verdict}")

## Download
Zips the results folder (runs, logs, pretraining receipts, analysis tables, the 70% budget file). A commit keeps it in the Output tab:
download `results_pass11_core.zip` and send it with the notebook log.

In [ ]:
shutil.copy(EXTRA_PATH, f"{OUT_DIR}/budgets_extra_p70.json")
print(shutil.make_archive("/kaggle/working/results_pass11_core", "zip", OUT_DIR))
print(f"notebook time: {(time.time() - T_START) / 60:.1f} min")